# 🚀 Advanced Offline Data Science RAG Chatbot
Full-featured classroom + interview preparation bot (works in Google Colab offline mode)

In [ ]:
!pip install sentence-transformers faiss-cpu transformers accelerate rank-bm25 gradio

In [ ]:

documents = [
# Add more detailed structured knowledge
"Pandas groupby is used to aggregate data based on categories such as sum mean and count.",
"SQL joins combine multiple tables based on common keys such as inner join left join and right join.",
"Overfitting happens when model memorizes training data and fails on unseen data.",
"Underfitting happens when model is too simple to learn patterns.",
"Precision is TP divided by TP plus FP.",
"Recall is TP divided by TP plus FN.",
"Gradient descent minimizes loss function using iterative updates.",
"Feature scaling ensures all features contribute equally to model training.",
"TF IDF measures importance of a word in a document relative to corpus.",
"Cross validation improves model robustness by multiple splits.",
"Random Forest reduces variance using multiple trees.",
"XGBoost improves performance using boosting technique.",
"Neural networks consist of layers input hidden and output.",
"Tokenization splits sentences into words or tokens.",
"Normalization scales values between 0 and 1.",
"Standardization transforms data to mean 0 and std 1."
]


In [ ]:

from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedding_model.encode(documents)


In [ ]:

import faiss

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))


In [ ]:

from rank_bm25 import BM25Okapi

tokenized_docs = [doc.split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

def hybrid_search(query, top_k=3):
    query_tokens = query.split()
    bm25_scores = bm25.get_scores(query_tokens)

    query_embedding = embedding_model.encode([query])
    _, faiss_indices = index.search(np.array(query_embedding), top_k)

    combined = list(faiss_indices[0]) + list(np.argsort(bm25_scores)[-top_k:])
    combined = list(set(combined))

    return [documents[i] for i in combined]


In [ ]:

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=200
)


In [ ]:

chat_history = []

def build_prompt(query, context):
    history = "\n".join(chat_history[-3:])
    return f"""
You are an expert Data Science tutor.

Conversation History:
{history}

Context:
{context}

Question:
{query}

Give structured answer with:
1. Simple explanation
2. Example
3. Key points
"""


In [ ]:

def chatbot(query):
    context = hybrid_search(query)
    prompt = build_prompt(query, context)

    response = generator(prompt)[0]['generated_text']

    chat_history.append(f"Q: {query}")
    chat_history.append(f"A: {response}")

    return response


In [ ]:

import random

def quiz_mode():
    q = random.choice(documents)
    print("Explain this concept:", q)


In [ ]:

import gradio as gr

def gradio_chat(query):
    return chatbot(query)

iface = gr.Interface(fn=gradio_chat, inputs="text", outputs="text", title="Data Science Tutor Bot")
iface.launch()


In [ ]:

while True:
    q = input("Ask (or type quiz/exit): ")

    if q == "exit":
        break
    elif q == "quiz":
        quiz_mode()
    else:
        print(chatbot(q))
